In [7]:
from __future__ import annotations

import os
import time
import importlib
from dataclasses import dataclass
from typing import Callable, Dict, List, Optional, Tuple

import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from scipy.sparse import hstack
from transformers import AutoModelForSequenceClassification, AutoTokenizer, EarlyStoppingCallback, Trainer, TrainingArguments


BASE_DIR = r'D:\Github\4021-Information-Retrieval\classification'
EVAL_PATH = os.path.join(BASE_DIR, 'evaluation', 'evaluation_dataset.xlsx')
PREPROCESSED_PATH = os.path.join(BASE_DIR, '..', 'preprocessing', 'preprocessed_data.csv')
NER_PATH = os.path.join(BASE_DIR, '..', 'preprocessing', 'NER_preprocessed_data.csv')
RESULTS_DIR = os.path.join(BASE_DIR, 'results')
EXCEL_OUTPUT_PATH = os.path.join(RESULTS_DIR, 'evaluation_results.xlsx')
COMPARISON_IMAGE_PREPROCESSED_PATH = os.path.join(RESULTS_DIR, 'model_comparison_preprocessed_text.png')
COMPARISON_IMAGE_NER_PATH = os.path.join(RESULTS_DIR, 'model_comparison_ner_text.png')
COMPARISON_IMAGE_HYBRID_PATH = os.path.join(RESULTS_DIR, 'model_comparison_hybrid_text.png')

os.makedirs(RESULTS_DIR, exist_ok=True)

TRANSFORMER_MODEL_NAME = 'distilbert-base-uncased-finetuned-sst-2-english'
NER_MODEL_NAME = 'en_core_web_sm'
MAX_LENGTH = 128
RANDOM_STATE = 99
TRANSFORMER_VAL_SIZE = 0.15
ALLOWED_NER_LABELS = {'PERSON', 'ORG', 'WORK_OF_ART', 'PRODUCT', 'EVENT'}

In [8]:
class EncodedTextWithSarcasmDataset(torch.utils.data.Dataset):
    def __init__(
        self,
        texts: pd.Series,
        labels: pd.Series,
        sarcasm_label: pd.Series,
        sarcasm_score: pd.Series,
        tokenizer: AutoTokenizer,
    ):
        self.encodings = tokenizer(
            texts.fillna('').tolist(),
            truncation=True,
            padding='max_length',
            max_length=MAX_LENGTH,
        )
        self.labels = labels.tolist()

        sarcasm_label_bin = (sarcasm_label.astype(str).str.lower() == "irony").astype(float)
        sarcasm_score = sarcasm_score.astype(float)

        self.extra_features = np.stack(
            [sarcasm_label_bin.to_numpy(), sarcasm_score.to_numpy()],
            axis=1
        )

    def __getitem__(self, idx: int):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["extra_features"] = torch.tensor(self.extra_features[idx], dtype=torch.float)
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

In [9]:
import torch
import torch.nn as nn
from transformers import AutoModel

class DistilBertWithSarcasmFeatures(nn.Module):
    def __init__(self, model_name: str, num_labels: int = 2, extra_feat_dim: int = 2):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size

        self.dropout = nn.Dropout(0.2)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size + extra_feat_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_labels)
        )

    def forward(self, input_ids=None, attention_mask=None, extra_features=None, labels=None):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        # DistilBERT has no pooler, so use CLS token representation
        cls_output = outputs.last_hidden_state[:, 0, :]   # [batch, hidden_size]

        combined = torch.cat([cls_output, extra_features], dim=1)
        logits = self.classifier(self.dropout(combined))

        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(logits, labels)

        return {"loss": loss, "logits": logits}

In [10]:
@dataclass(frozen=True)
class ModelSpec:
    name: str
    kind: str
    factory: Optional[Callable[[], object]] = None


MODEL_SPECS: List[ModelSpec] = [
    ModelSpec('Logistic Regression', 'classical', lambda: LogisticRegression(max_iter=200, class_weight='balanced')),
    ModelSpec('SVM', 'classical', lambda: SVC(kernel='linear', class_weight='balanced', probability=True, random_state=RANDOM_STATE)),
    ModelSpec('DummyClassifier', 'classical', lambda: DummyClassifier(strategy='uniform', random_state=RANDOM_STATE)),
    ModelSpec('Transformer (DistilBERT)', 'transformer'),
]

TEXT_SOURCES: List[Tuple[str, str]] = [
    ('preprocessed_text', PREPROCESSED_PATH),
    ('augmented_text', ''),
]


class EncodedTextDataset(torch.utils.data.Dataset):
    def __init__(self, texts: pd.Series, labels: pd.Series, tokenizer: AutoTokenizer):
        self.encodings = tokenizer(
            texts.tolist(),
            truncation=True,
            padding='max_length',
            max_length=MAX_LENGTH,
        )
        self.labels = labels.tolist()

    def __getitem__(self, idx: int):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self) -> int:
        return len(self.labels)
    
def ensure_review_id(df: pd.DataFrame) -> pd.DataFrame:
    if 'review_id' not in df.columns:
        df = df.reset_index().rename(columns={'index': 'review_id'})
    return df


def load_evaluation_split() -> Tuple[pd.DataFrame, pd.DataFrame]:
    eval_df = pd.read_excel(EVAL_PATH)
    eval_df = ensure_review_id(eval_df)

    eval_df['ground_truth_polarity'] = (
        eval_df['ground_truth_polarity']
        .astype(str)
        .str.strip()
        .str.upper()
    )

    eval_df['polarity'] = eval_df['ground_truth_polarity'].map({
        'POSITIVE': 1,
        'NEGATIVE': 0,
        'NEUTRAL': 2,
    })

    df_pol = eval_df[eval_df['polarity'] != 2].copy()

    train_meta, test_meta = train_test_split(
        df_pol[['review_id', 'ground_truth_polarity', 'polarity']],
        test_size=0.2,
        stratify=df_pol['polarity'],
        random_state=RANDOM_STATE,
    )

    return train_meta.copy(), test_meta.copy()


def load_text_source(path: str, output_column: str) -> pd.DataFrame:
    source_df = pd.read_csv(path, usecols=['review_id', 'processed_text'])
    source_df = ensure_review_id(source_df)
    source_df = source_df.rename(columns={'processed_text': output_column})
    return source_df[['review_id', output_column]].copy()


def load_ner_pipeline():
    try:
        spacy = importlib.import_module('spacy')
    except ImportError as exc:
        raise RuntimeError(
            "spaCy is not installed. Install with: pip install spacy"
        ) from exc

    try:
        return spacy.load(NER_MODEL_NAME, disable=['parser', 'lemmatizer', 'textcat'])
    except OSError as exc:
        raise RuntimeError(
            f"spaCy model '{NER_MODEL_NAME}' is not installed. "
            f"Install with: python -m spacy download {NER_MODEL_NAME}"
        ) from exc


def build_ner_text(series: pd.Series, nlp) -> pd.Series:
    entity_token_rows: List[str] = []
    for doc in nlp.pipe(series.fillna('').astype(str).tolist(), batch_size=64):
        tokens = []
        seen_pairs = set()
        for ent in doc.ents:
            if ent.label_ not in ALLOWED_NER_LABELS:
                continue
            # Encode both entity label and value as lexical features.
            text_token = ent.text.strip().replace(' ', '_')
            if text_token:
                token_pair = (ent.label_, text_token.lower())
                if token_pair in seen_pairs:
                    continue
                seen_pairs.add(token_pair)
                tokens.append(f'ENTLBL_{ent.label_}')
                tokens.append(f'ENTTXT_{text_token.lower()}')
        entity_token_rows.append(' '.join(tokens))
    return pd.Series(entity_token_rows, index=series.index)


def build_shared_datasets(train_meta: pd.DataFrame, test_meta: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    preprocessed_df = load_text_source(PREPROCESSED_PATH, 'preprocessed_text')

    train_df = (
        train_meta
        .merge(preprocessed_df, on='review_id', how='left')
    )
    test_df = (
        test_meta
        .merge(preprocessed_df, on='review_id', how='left')
    )

    nlp = load_ner_pipeline()
    train_df['ner_text'] = build_ner_text(train_df['preprocessed_text'], nlp)
    test_df['ner_text'] = build_ner_text(test_df['preprocessed_text'], nlp)

    train_df['ner_text'] = train_df['ner_text'].fillna('')
    test_df['ner_text'] = test_df['ner_text'].fillna('')
    train_df['augmented_text'] = (train_df['preprocessed_text'].fillna('') + ' [SEP] ' + train_df['ner_text']).str.strip()
    test_df['augmented_text'] = (test_df['preprocessed_text'].fillna('') + ' [SEP] ' + test_df['ner_text']).str.strip()

    missing_train = train_df['preprocessed_text'].isna().sum()
    missing_test = test_df['preprocessed_text'].isna().sum()
    if missing_train or missing_test:
        raise ValueError(
            f'Missing preprocessed_text values: train={missing_train}, test={missing_test}. '
            'Check that review_id matches across the files.'
        )

    return train_df, test_df


def classical_model_report(
    model_name: str,
    model_factory: Callable[[], object],
    x_train_vec,
    y_train: pd.Series,
    x_test_vec,
    y_test: pd.Series,
) -> Tuple[pd.DataFrame, Dict[str, object], pd.Series]:
    model = model_factory()
    model.fit(x_train_vec, y_train)

    start_time = time.perf_counter()
    y_pred = model.predict(x_test_vec)
    classification_time = time.perf_counter() - start_time

    report = classification_report(
        y_test,
        y_pred,
        digits=3,
        output_dict=True,
        zero_division=0,
    )

    report_df = pd.DataFrame(report).transpose().reset_index().rename(columns={'index': 'label'})
    report_df['label'] = report_df['label'].replace({
        '0': 'NEGATIVE',
        '1': 'POSITIVE',
        0: 'NEGATIVE',
        1: 'POSITIVE',
    })
    summary_row = {
        'model': model_name,
        'accuracy': report.get('accuracy', 0.0),
        'macro_f1': report['macro avg']['f1-score'],
        'weighted_f1': report['weighted avg']['f1-score'],
        'classification_time_seconds': classification_time,
    }
    predictions = pd.Series(y_pred, index=y_test.index)
    return report_df, summary_row, predictions


def transformer_model_report(
    model_name: str,
    x_train: pd.Series,
    y_train: pd.Series,
    x_test: pd.Series,
    y_test: pd.Series,
    sarcasm_label_train: pd.Series,
    sarcasm_score_train: pd.Series,
    sarcasm_label_test: pd.Series,
    sarcasm_score_test: pd.Series,
):
    tokenizer = AutoTokenizer.from_pretrained(TRANSFORMER_MODEL_NAME)

    stratify_labels = y_train if y_train.nunique() > 1 and y_train.value_counts().min() >= 2 else None
    (
        x_train_fit, x_val,
        y_train_fit, y_val,
        sarcasm_label_train_fit, sarcasm_label_val,
        sarcasm_score_train_fit, sarcasm_score_val
    ) = train_test_split(
        x_train.fillna(''),
        y_train,
        sarcasm_label_train,
        sarcasm_score_train,
        test_size=TRANSFORMER_VAL_SIZE,
        random_state=RANDOM_STATE,
        stratify=stratify_labels,
    )

    print('HELLO')

    train_dataset = EncodedTextWithSarcasmDataset(
        x_train_fit, y_train_fit,
        sarcasm_label_train_fit, sarcasm_score_train_fit,
        tokenizer
    )
    val_dataset = EncodedTextWithSarcasmDataset(
        x_val, y_val,
        sarcasm_label_val, sarcasm_score_val,
        tokenizer
    )
    test_dataset = EncodedTextWithSarcasmDataset(
        x_test.fillna(''), y_test,
        sarcasm_label_test, sarcasm_score_test,
        tokenizer
    )

    model = DistilBertWithSarcasmFeatures(
        model_name=TRANSFORMER_MODEL_NAME,
        num_labels=2,
        extra_feat_dim=2
    )

    training_args = TrainingArguments(
        output_dir=os.path.join(RESULTS_DIR, 'transformer_runs', model_name.replace(' ', '_').lower()),
        num_train_epochs=3,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        learning_rate=2e-5,
        warmup_ratio=0.1,
        weight_decay=0.01,
        eval_strategy='epoch',
        logging_strategy='epoch',
        save_strategy='epoch',
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        greater_is_better=False,
        report_to=[],
        seed=RANDOM_STATE,
        fp16=torch.cuda.is_available(),
        disable_tqdm=True,
        remove_unused_columns=False,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    trainer.train()


    start_time = time.perf_counter()
    prediction_output = trainer.predict(test_dataset)
    classification_time = time.perf_counter() - start_time

    y_pred = prediction_output.predictions.argmax(axis=1)
    
    report = classification_report(
        y_test,
        y_pred,
        digits=3,
        output_dict=True,
        zero_division=0,
    )

    report_df = pd.DataFrame(report).transpose().reset_index().rename(columns={'index': 'label'})
    report_df['label'] = report_df['label'].replace({
        '0': 'NEGATIVE',
        '1': 'POSITIVE',
        0: 'NEGATIVE',
        1: 'POSITIVE',
    })
    summary_row = {
        'model': model_name,
        'accuracy': report.get('accuracy', 0.0),
        'macro_f1': report['macro avg']['f1-score'],
        'weighted_f1': report['weighted avg']['f1-score'],
        'classification_time_seconds': classification_time,
    }
    predictions = pd.Series(y_pred, index=y_test.index)
    return report_df, summary_row, predictions


def evaluate_text_source(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    text_column: str,
) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, pd.Series]]:
    x_train = train_df[text_column].fillna('')
    y_train = train_df['polarity']
    x_test = test_df[text_column].fillna('')
    y_test = test_df['polarity']

    vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
    x_train_vec = vectorizer.fit_transform(x_train)
    x_test_vec = vectorizer.transform(x_test)

    detailed_rows: List[pd.DataFrame] = []
    summary_rows: List[Dict[str, object]] = []
    predictions: Dict[str, pd.Series] = {}

    for spec in MODEL_SPECS:
        if spec.kind == 'classical':
            if spec.factory is None:
                raise ValueError(f'Missing factory for classical model {spec.name}')
            report_df, summary_row, y_pred = classical_model_report(
                spec.name,
                spec.factory,
                x_train_vec,
                y_train,
                x_test_vec,
                y_test,
            )
        elif spec.kind == 'transformer':
            report_df, summary_row, y_pred = transformer_model_report(
                spec.name,
                x_train,
                y_train,
                x_test,
                y_test,
            )
        else:
            raise ValueError(f'Unknown model kind: {spec.kind}')

        report_df.insert(0, 'text_source', text_column)
        report_df.insert(1, 'model', spec.name)
        detailed_rows.append(report_df)

        summary_row['text_source'] = text_column
        summary_rows.append(summary_row)

        predictions[spec.name] = y_pred

    detailed_report_df = pd.concat(detailed_rows, ignore_index=True)
    summary_df = pd.DataFrame(summary_rows)
    return detailed_report_df, summary_df, predictions


def choose_best_model(summary_df: pd.DataFrame, text_source: str) -> str:
    source_summary = summary_df[summary_df['text_source'] == text_source].copy()
    best_row = source_summary.sort_values(['weighted_f1', 'accuracy'], ascending=False).iloc[0]
    return str(best_row['model'])


def save_comparison_table_image(summary_df: pd.DataFrame, text_source: str, output_path: str) -> None:
    display_df = summary_df[summary_df['text_source'] == text_source].copy()
    best_model = choose_best_model(summary_df, text_source)
    display_df['selected_model'] = display_df['model'] == best_model

    display_df = display_df.sort_values(['weighted_f1', 'accuracy'], ascending=[False, False]).reset_index(drop=True)
    display_df['classification_time_seconds'] = display_df['classification_time_seconds'].map(lambda value: f'{value:.4f}')
    display_df['accuracy'] = display_df['accuracy'].map(lambda value: f'{value:.3f}')
    display_df['macro_f1'] = display_df['macro_f1'].map(lambda value: f'{value:.3f}')
    display_df['weighted_f1'] = display_df['weighted_f1'].map(lambda value: f'{value:.3f}')

    table_df = display_df[['model', 'accuracy', 'macro_f1', 'weighted_f1', 'classification_time_seconds']].copy()

    fig_height = max(4.5, 0.7 * (len(table_df) + 2))
    fig, ax = plt.subplots(figsize=(16, fig_height))
    ax.axis('off')

    table = ax.table(
        cellText=table_df.values,
        colLabels=table_df.columns,
        loc='center',
        cellLoc='center',
        colLoc='center',
    )

    table.auto_set_font_size(False)
    table.set_fontsize(11)
    table.scale(1.2, 1.5)

    selected_rows = display_df.index[display_df['selected_model']].tolist()
    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_text_props(weight='bold', color='white')
            cell.set_facecolor('#264653')
        else:
            data_row = row - 1
            if data_row in selected_rows:
                cell.set_facecolor('#c8f7c5')
                if col == 1:
                    cell.set_text_props(weight='bold')
            elif data_row % 2 == 0:
                cell.set_facecolor('#f7f7f7')
            else:
                cell.set_facecolor('#ebf2f7')

    title_label_map = {
        'preprocessed_text': 'preprocessed_text',
        'ner_text': 'ner_text',
        'hybrid_text': 'hybrid_text (preprocessed_text + ner_text)',
    }
    title_text = f"Model Comparison Summary - {title_label_map.get(text_source, text_source)}"
    plt.title(title_text, fontsize=18, weight='bold', pad=20)
    plt.tight_layout()
    plt.savefig(output_path, dpi=220, bbox_inches='tight')
    plt.close(fig)


In [13]:
import polars as pl

In [14]:
train_meta, test_meta = load_evaluation_split()
train_df, test_df = build_shared_datasets(train_meta, test_meta)
temp = pl.read_csv(r'D:\Github\4021-Information-Retrieval\classification\results\test_dataset_results.csv').to_pandas()
common = test_df["review_id"].isin(temp["review_id"])
print(common.sum())

test_dataset = test_df[['review_id', 'ground_truth_polarity', 'preprocessed_text', 'ner_text', 'augmented_text']].copy()

comparison_tables: List[pd.DataFrame] = []
summary_tables: List[pd.DataFrame] = []
prediction_store: Dict[Tuple[str, str], pd.Series] = {}

200


### Making use of a pretrained sarcasm model

In [17]:
from transformers import pipeline

sarcasm_model = pipeline(
    "text-classification",
    model="cardiffnlp/twitter-roberta-base-irony"
)

def get_sarcasm(text):
    if text is None or str(text).strip() == "":
        return {"label": None, "score": None}
    
    result = sarcasm_model(text[:512])[0]  # truncate long reviews
    return result

Device set to use cuda:0


In [18]:
sarcasm_train = train_df["preprocessed_text"].apply(get_sarcasm)
sarcasm_test = test_df["preprocessed_text"].apply(get_sarcasm)

train_df["preprocessed_text_sarcasm_label"] = sarcasm_train.apply(lambda x: x["label"])
train_df["preprocessed_text_sarcasm_score"] = sarcasm_train.apply(lambda x: x["score"])

test_df["preprocessed_text_sarcasm_label"] = sarcasm_train.apply(lambda x: x["label"])
test_df["preprocessed_text_sarcasm_score"] = sarcasm_train.apply(lambda x: x["score"])

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [20]:
sarcasm_train = train_df["augmented_text"].apply(get_sarcasm)
sarcasm_test = test_df["augmented_text"].apply(get_sarcasm)

train_df["augmented_text_sarcasm_label"] = sarcasm_train.apply(lambda x: x["label"])
train_df["augmented_text_sarcasm_score"] = sarcasm_train.apply(lambda x: x["score"])

test_df["augmented_text_sarcasm_label"] = sarcasm_train.apply(lambda x: x["label"])
test_df["augmented_text_sarcasm_score"] = sarcasm_train.apply(lambda x: x["score"])

### Training and Evaluation

In [23]:
import numpy as np

In [24]:
for text_source, _ in TEXT_SOURCES:
    if text_source == 'ner_text':
        continue
    x_train = train_df[text_source].fillna('')
    y_train = train_df['polarity']
    x_test = test_df[text_source].fillna('')
    y_test = test_df['polarity']

    detailed_rows: List[pd.DataFrame] = []
    summary_rows: List[Dict[str, object]] = []
    predictions: Dict[str, pd.Series] = {}

    for spec in MODEL_SPECS:
        if spec.kind == 'classical':
            continue
        elif spec.kind == 'transformer':
            report_df, summary_row, y_pred = transformer_model_report(
                spec.name,
                x_train,
                y_train,
                x_test,
                y_test,
                train_df[f"{text_source}_sarcasm_label"],
                train_df[f"{text_source}_sarcasm_score"],
                test_df[f"{text_source}_sarcasm_label"],
                test_df[f"{text_source}_sarcasm_score"],
            )
        else:
            raise ValueError(f'Unknown model kind: {spec.kind}')
        
        report_df.insert(0, 'text_source', text_source)
        report_df.insert(1, 'model', spec.name)
        detailed_rows.append(report_df)

        summary_row['text_source'] = text_source
        summary_rows.append(summary_row)

        predictions[spec.name] = y_pred

    for model_name, series in predictions.items():
        prediction_store[(text_source, model_name)] = series

    detailed_report_df = pd.concat(detailed_rows, ignore_index=True)
    summary_df = pd.DataFrame(summary_rows)

    comparison_tables.append(detailed_report_df)
    summary_tables.append(summary_df)

HELLO
{'loss': 0.5089, 'grad_norm': 5.605478763580322, 'learning_rate': 1.5021834061135372e-05, 'epoch': 1.0}
{'eval_loss': 0.5241811275482178, 'eval_runtime': 0.1652, 'eval_samples_per_second': 726.489, 'eval_steps_per_second': 90.811, 'epoch': 1.0}
{'loss': 0.2919, 'grad_norm': 22.96244239807129, 'learning_rate': 7.685589519650655e-06, 'epoch': 2.0}
{'eval_loss': 0.5380694270133972, 'eval_runtime': 0.1447, 'eval_samples_per_second': 829.093, 'eval_steps_per_second': 103.637, 'epoch': 2.0}
{'loss': 0.1585, 'grad_norm': 0.3670424818992615, 'learning_rate': 2.6200873362445414e-07, 'epoch': 3.0}
{'eval_loss': 0.7266693711280823, 'eval_runtime': 0.1335, 'eval_samples_per_second': 898.787, 'eval_steps_per_second': 112.348, 'epoch': 3.0}
{'train_runtime': 30.4474, 'train_samples_per_second': 67.001, 'train_steps_per_second': 8.375, 'train_loss': 0.31976315367455577, 'epoch': 3.0}
HELLO
{'loss': 0.4738, 'grad_norm': 5.729396343231201, 'learning_rate': 1.5021834061135372e-05, 'epoch': 1.0}
{'

In [25]:
print(prediction_store.keys())

dict_keys([('preprocessed_text', 'Transformer (DistilBERT)'), ('augmented_text', 'Transformer (DistilBERT)')])


In [26]:
df = pl.read_csv(r'D:\Github\4021-Information-Retrieval\classification\results\test_dataset_results.csv')

df = df.with_columns(
    pl.col("ground_truth_polarity")
    .str.to_uppercase()
    .replace({
        "NEGATIVE": 0,
        "POSITIVE": 1
    })
    .alias("gt_label")
)

df = df.with_columns(
    pl.col("gt_label").cast(pl.Int64)
)

In [27]:
gt = df["gt_label"].to_numpy()

# check exact match (order + values)
np.array_equal(gt, y_test)

True

In [28]:
preds = prediction_store[('preprocessed_text', 'Transformer (DistilBERT)')]
pred_labels = np.where(preds == 1, "POSITIVE", "NEGATIVE")

df = df.with_columns(
    pl.Series("preprocessed_sarcasm_prediction_labels", pred_labels)
)

preds = prediction_store[('augmented_text', 'Transformer (DistilBERT)')]
pred_labels = np.where(preds == 1, "POSITIVE", "NEGATIVE")

df = df.with_columns(
    pl.Series("augmented_sarcasm_prediction_labels", pred_labels)
)

df.write_csv('test_sarcasm_result.csv')

In [29]:
comparison_tables[0]

,text_source,model,label,precision,recall,f1-score,support
0,preprocessed_text,Transformer (DistilBERT),NEGATIVE,0.810606,0.938596,0.869919,114.00
1,preprocessed_text,Transformer (DistilBERT),POSITIVE,0.897059,0.709302,0.792208,86.00
2,preprocessed_text,Transformer (DistilBERT),accuracy,0.840000,0.840000,0.840000,0.84
3,preprocessed_text,Transformer (DistilBERT),macro avg,0.853832,0.823949,0.831063,200.00
4,preprocessed_text,Transformer (DistilBERT),weighted avg,0.847781,0.840000,0.836503,200.00


In [30]:
comparison_tables[1]

,text_source,model,label,precision,recall,f1-score,support
0,augmented_text,Transformer (DistilBERT),NEGATIVE,0.843750,0.947368,0.892562,114.00
1,augmented_text,Transformer (DistilBERT),POSITIVE,0.916667,0.767442,0.835443,86.00
2,augmented_text,Transformer (DistilBERT),accuracy,0.870000,0.870000,0.870000,0.87
3,augmented_text,Transformer (DistilBERT),macro avg,0.880208,0.857405,0.864003,200.00
4,augmented_text,Transformer (DistilBERT),weighted avg,0.875104,0.870000,0.868001,200.00


In [31]:
comparison_tables_all = pd.concat(comparison_tables, ignore_index=True)
summary_tables_all = pd.concat(summary_tables, ignore_index=True)

In [35]:
pl.from_pandas(comparison_tables_all).write_csv('comparison_tables.csv')
comparison_tables_all

,text_source,model,label,precision,recall,f1-score,support
0,preprocessed_text,Transformer (DistilBERT),NEGATIVE,0.810606,0.938596,0.869919,114.00
1,preprocessed_text,Transformer (DistilBERT),POSITIVE,0.897059,0.709302,0.792208,86.00
2,preprocessed_text,Transformer (DistilBERT),accuracy,0.840000,0.840000,0.840000,0.84
3,preprocessed_text,Transformer (DistilBERT),macro avg,0.853832,0.823949,0.831063,200.00
4,preprocessed_text,Transformer (DistilBERT),weighted avg,0.847781,0.840000,0.836503,200.00
5,augmented_text,Transformer (DistilBERT),NEGATIVE,0.843750,0.947368,0.892562,114.00
6,augmented_text,Transformer (DistilBERT),POSITIVE,0.916667,0.767442,0.835443,86.00
7,augmented_text,Transformer (DistilBERT),accuracy,0.870000,0.870000,0.870000,0.87
8,augmented_text,Transformer (DistilBERT),macro avg,0.880208,0.857405,0.864003,200.00
9,augmented_text,Transformer (DistilBERT),weighted avg,0.875104,0.870000,0.868001,200.00


In [36]:
pl.from_pandas(summary_tables_all).write_csv('summary_tables.csv')
summary_tables_all

,model,accuracy,macro_f1,weighted_f1,classification_time_seconds,text_source
0,Transformer (DistilBERT),0.84,0.831063,0.836503,0.314692,preprocessed_text
1,Transformer (DistilBERT),0.87,0.864003,0.868001,0.316716,augmented_text


Preprocessed Text

| Metric                  | Baseline (No Sarcasm) | + Sarcasm | Difference |
| ----------------------- | --------------------- | --------- | ---------- |
| **Accuracy**            | 0.865                 | 0.865     | 0.000      |
| **Precision (Macro)**   | 0.876                 | 0.866     | -0.010     |
| **Recall (Macro)**      | 0.852                 | 0.857     | +0.005     |
| **F1-score (Macro)**    | 0.858                 | 0.861     | +0.003     |
| **F1-score (Weighted)** | 0.863                 | 0.864     | +0.001     |


Post NER

| Metric                  | Baseline (No Sarcasm) | + Sarcasm | Difference |
| ----------------------- | ----------------------- | --------- | ---------- |
| **Accuracy**            | 0.870                   | 0.865     | **-0.005** |
| **Precision (Macro)**   | 0.877                   | 0.866     | **-0.011** |
| **Recall (Macro)**      | 0.859                   | 0.857     | **-0.002** |
| **F1-score (Macro)**    | 0.865                   | 0.861     | **-0.004** |
| **F1-score (Weighted)** | 0.868                   | 0.864     | **-0.004** |
